# Procesamiento Fiscal · `Fiscal-<Depto>_AUTO.xlsx` para Loreto, San Martín y Ucayali

Lee todo lo que producen los scrapers en `Fiscal/Scrapeo/output/` más el `IPC.xlsx` de
`Fiscal/Procesamiento/input/`, y genera **un workbook nuevo por departamento** en
`Fiscal/Procesamiento/output/Fiscal-<Depto>_AUTO.xlsx` con la misma estructura que el
workbook manual (series mensuales en miles de soles, columnas Total por año, bloques
reales deflactados con IPC, variación acumulada Ene.-Mes vs año anterior):

| Hoja | Bloques | Input que la alimenta |
|---|---|---|
| `GASTO G.N.` / `GASTOS G.R.` / `GASTOS G.L.` | Gasto por genérica, servicio de la deuda, gasto por fuente, bloque real | `DEV Generica.*`, `DEV Deuda.*`, `DEV Fuente.*` |
| `INVERSIÓN G.N.` / `G.R.` / `G.L.` | Inversión (2.6) por función y por fuente | `DEV Funcion.*`, `DEV InvFuente.*` |
| `PIM G.N.` / `G.R.` / `G.L.` | PIM anual por función, devengado 2.6 acumulado y % de avance | `PIM Funcion*`, `DEV Funcion.*` |
| `CONTROL` | Estado de todos los CSV de control (ESTADO = OK / PARCIAL / REVISAR) | `CARPETA DE CONTROL`, `PIM/*/CONTROL` |

Además, el **Bloque 6** genera `Variaciones alarmantes AAAA-MM.xlsx` en la carpeta de Drive `VARIACIONES_FOLDER_ID`: el total
de inversión de cada nivel de gobierno y de los tres juntos frente al periodo anterior, y los 5
mayores aumentos y las 5 mayores caídas de la inversión (2.6) **en soles corrientes, no en %**, por departamento y nivel
de gobierno (por función; en G.L. también por municipalidad) y entre todos los niveles juntos. Compara el último mes con
el mismo mes del año anterior y el acumulado Ene.-Mes con el del año anterior.

Si falta un input (por ejemplo aún no corriste el scraper de genéricas), el bloque
correspondiente queda vacío y se avisa en el resumen; nada se inventa.

In [ ]:
# ═══ BLOQUE 1 · LIBRERÍAS Y DRIVE ═══
!pip install openpyxl pandas numpy --quiet
import os, re, glob, unicodedata, warnings
from datetime import date, datetime
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter as L
from IPython.display import display
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount("/content/drive")
RUTA_FISCAL = "/content/drive/MyDrive/Fiscal"
assert os.path.isdir(RUTA_FISCAL), f"No encuentro {RUTA_FISCAL}"
print("✅ Drive montado")


In [ ]:
# ═══ BLOQUE 2 · CONFIGURACIÓN ═══
RUTA_SCRAPEO = os.path.join(RUTA_FISCAL, "Scrapeo", "output")
RUTA_IPC     = os.path.join(RUTA_FISCAL, "Procesamiento", "input", "IPC.xlsx")
RUTA_SALIDA  = os.path.join(RUTA_FISCAL, "Procesamiento", "output")
os.makedirs(RUTA_SALIDA, exist_ok=True)

DEPARTAMENTOS = {"16": ("LORETO", "Loreto"), "22": ("SAN MARTÍN", "SanMartin"), "25": ("UCAYALI", "Ucayali")}
AÑO_INICIO = 2025                 # primer año de las series (los scrapers empiezan en 2025)
IPC_BASE_PROM_ANUAL = 2007        # base del deflactor: IPC_t / promedio(IPC año base). El workbook manual usa 2007 (≈63.78)
IPC_BASE_FIJO = 63.78             # se usa si el IPC.xlsx no cubre el año base (empieza en 2010)
SUFIJO_SALIDA = "_AUTO"           # Fiscal-Loreto_AUTO.xlsx (no toca el archivo manual)
AGRUPAR_COMUNICACIONES_EN_TRANSPORTE = True   # convención del workbook manual: 16: Comunicaciones se suma en 15: Transporte

# ── Variaciones alarmantes (Bloque 6) ──
VARIACIONES_FOLDER_ID = "1LgIhySIoywZDMTGPyD9rvjiBHc1FYvDl"   # carpeta de Drive donde se crea "Variaciones alarmantes AAAA-MM.xlsx"
VARIACIONES_NOMBRE = "Variaciones alarmantes"
VARIACIONES_TOP = 5               # cuántos aumentos y cuántas caídas (en soles, no en %) por bloque
VARIACIONES_COMPARACIONES = ["MES_AÑO_ANTERIOR", "ACUMULADO"]   # opción adicional: "MES_ANTERIOR" (mes vs mes anterior)

NIVELES = [  # (corto en nombres de archivo, carpeta DEVENGADO, etiqueta PIM, nombre hoja gasto, nombre hoja inversión, sufijo PIM, título)
    dict(corto="GobNacional",  carpeta="Gobierno Nacional", pim="GOBIERNO NACIONAL",    hoja_g="GASTO G.N.",  hoja_i="INVERSIÓN G.N.", hoja_p="PIM G.N.", nombre="GOBIERNO NACIONAL"),
    dict(corto="GobRegional",  carpeta="Gobierno Regional", pim="GOBIERNOS REGIONALES", hoja_g="GASTOS G.R.", hoja_i="INVERSIÓN G.R.", hoja_p="PIM G.R.", nombre="GOBIERNO REGIONAL"),
    dict(corto="Municipalidad",carpeta="Gobierno Municipal",pim="GOBIERNOS LOCALES",    hoja_g="GASTOS G.L.", hoja_i="INVERSIÓN G.L.", hoja_p="PIM G.L.", nombre="GOBIERNOS LOCALES"),
]

# Filas del bloque por genérica (como en el workbook manual). Cada fila se identifica por la ETIQUETA que
# devuelve el MEF (no por el código, que cambia de formato) y por el tipo de transacción: 5 corriente / 6 capital / 7 deuda.
GENERICAS = [
    ("GASTOS CORRIENTES", None),
    ("Personal y obligaciones",               ("5", r"personal")),
    ("Pensiones y prestaciones sociales",     ("5", r"pension")),
    ("Bienes y servicios",                    ("5", r"bienes y servicios")),
    ("Donaciones y transferencias",           ("5", r"donaciones")),
    ("Otros gastos corrientes",               ("5", r"otros gastos")),
    ("GASTOS DE CAPITAL", None),
    ("Donaciones y transferencias (2)",       ("6", r"donaciones")),
    ("Adquisición de activos no financieros", ("6", r"activos no financieros")),
    ("Inversiones financieras",               ("6", r"activos financieros")),
    ("Otros gastos de capital",               ("6", r"otros gastos")),
    ("SERVICIO DE LA DEUDA", None),
    ("Intereses+comisiones",                  "DEUDA:INTERESES Y COMISIONES"),
    ("Amortización",                          "DEUDA:AMORTIZACION"),
]
# Filas del bloque por fuente, identificadas por la etiqueta de la fuente
FUENTES = [
    ("Recursos Ordinarios (Asignación MEF)",      r"recursos ordinarios"),
    ("Recursos determinados",                     r"recursos determinados"),
    ("Recursos Directamente Recaudados",          r"directamente recaudados"),
    ("Recursos por Oper. Ofi. De Créd. Interno",  r"operaciones oficiales de credito"),
    ("Donaciones y Transferencias",               r"donaciones"),
]
def normalizar(txt):
    import unicodedata
    t = unicodedata.normalize("NFKD", str(txt or ""))
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

def clasificar_generica(cod, etiqueta):
    """-> nombre de fila de GENERICAS o None. Tipo de transacción = primer tramo del código ('5-21' -> '5')."""
    tipo, t = str(cod).strip().split("-")[0], normalizar(etiqueta)
    t = re.sub(r"^[\w\-\.]+\s*:\s*", "", t)          # quita el código de la etiqueta
    for nombre, regla in GENERICAS:
        if isinstance(regla, tuple) and regla[0] == tipo and re.search(regla[1], t):
            return nombre
    return None

def clasificar_deuda(ruta):
    """Clasifica una fila del scraper de deuda por el texto de su RUTA (respaldo para CSV con CLASIFICACION = OTRO)."""
    t = normalizar(ruta)
    return "AMORTIZACION" if "amortiz" in t else "INTERESES Y COMISIONES" if ("interes" in t or "comision" in t) else "OTRO"

def clasificar_fuente(etiqueta):
    t = re.sub(r"^[\w\-\.]+\s*:\s*", "", normalizar(etiqueta))
    for nombre, regla in FUENTES:
        if re.search(regla, t):
            return nombre
    return None

FUNCIONES = ["01: LEGISLATIVA", "02: RELACIONES EXTERIORES", "03: PLANEAMIENTO, GESTION Y RESERVA DE CONTINGENCIA",
             "04: DEFENSA Y SEGURIDAD NACIONAL", "05: ORDEN PUBLICO Y SEGURIDAD", "06: JUSTICIA", "07: TRABAJO",
             "08: COMERCIO", "09: TURISMO", "10: AGROPECUARIA", "11: PESCA", "12: ENERGIA", "13: MINERIA",
             "14: INDUSTRIA", "15: TRANSPORTE", "16: COMUNICACIONES", "17: AMBIENTE", "18: SANEAMIENTO",
             "19: VIVIENDA Y DESARROLLO URBANO", "20: SALUD", "21: CULTURA Y DEPORTE", "22: EDUCACION",
             "23: PROTECCION SOCIAL", "24: PREVISION SOCIAL", "25: DEUDA PUBLICA"]


In [ ]:
# ═══ BLOQUE 3 · LECTURA DE INPUTS ═══
_RE_MES = re.compile(r"(\d{4}) - Mes (\d{2}) - ")

def leer_csvs(patron):
    """Concatena todos los CSV que cumplen el patrón (glob). Añade AÑO/MES_NUM desde el nombre si hace falta."""
    archivos = sorted(glob.glob(patron))
    partes = []
    for a in archivos:
        df = pd.read_csv(a, encoding="utf-8-sig")
        m = _RE_MES.search(os.path.basename(a))
        if m:
            df["AÑO"] = int(m.group(1)); df["MES_NUM"] = int(m.group(2))
        elif "MES" in df.columns:
            df["MES_NUM"] = df["MES"].astype(str).str.extract(r"^(\d+)")[0].astype(int)
        df["ARCHIVO"] = os.path.basename(a)
        partes.append(df)
    if not partes:
        return pd.DataFrame()
    df = pd.concat(partes, ignore_index=True)
    df["COD_DEPTO"] = df["DEPARTAMENTO"].astype(str).str.extract(r"^(\d+)")[0]
    if "MES_NUM" in df.columns:
        df["FECHA"] = pd.to_datetime(dict(year=df["AÑO"], month=df["MES_NUM"], day=1))
    return df

def codigo(serie):
    return serie.astype(str).str.extract(r"^([\w\-\.]+)\s*:")[0]

inputs, faltantes = {}, []
detalle_municipal = pd.DataFrame()   # DEV Funcion.Municipalidad por municipalidad (Bloque 6)
for n in NIVELES:
    carpeta = os.path.join(RUTA_SCRAPEO, "DEVENGADO", n["carpeta"])
    for dim in ["Funcion", "Generica", "Fuente", "InvFuente", "Deuda"]:
        df = leer_csvs(os.path.join(carpeta, f"DEV {dim}.{n['corto']} * - Mes * - *.csv"))
        inputs[(n["corto"], dim)] = df
        if df.empty:
            faltantes.append(f"DEV {dim}.{n['corto']}")
    # el scraper municipal trae el detalle por municipalidad: se agrega al departamento
    f = inputs[(n["corto"], "Funcion")]
    if not f.empty:
        f["COD_FUNCION"] = codigo(f["FUNCION"])
        if "MUNICIPALIDAD" in f.columns:
            detalle_municipal = f.copy()
        inputs[(n["corto"], "Funcion")] = (f.groupby(["COD_DEPTO", "FECHA", "AÑO", "MES_NUM", "COD_FUNCION", "FUNCION"], as_index=False)["DEVENGADO"].sum())

pim = leer_csvs(os.path.join(RUTA_SCRAPEO, "PIM", "PIM X FUNCIÓN", "DATA", "PIM Funcion * - *.csv"))
if not pim.empty:
    pim["COD_FUNCION"] = codigo(pim["FUNCION"])
else:
    faltantes.append("PIM Funcion")

controles = []
for a in sorted(glob.glob(os.path.join(RUTA_SCRAPEO, "**", "CONTROL*.csv"), recursive=True)):
    c = pd.read_csv(a, encoding="utf-8-sig")
    if "ESTADO" in c.columns:
        controles.append(pd.DataFrame({"ARCHIVO": os.path.basename(a), "AÑO": c.get("AÑO"), "DEPARTAMENTO": c.get("DEPARTAMENTO"),
                                       "ESTADO": c["ESTADO"], "FECHA_EXTRACCION": c.get("FECHA_EXTRACCION")}))
controles = pd.concat(controles, ignore_index=True) if controles else pd.DataFrame()

# IPC (INEI, Lima, Dic.2021=100): columnas Fecha | IPC a partir de la fila 4
ipc_raw = pd.read_excel(RUTA_IPC, sheet_name=0, header=None, skiprows=3, usecols=[1, 2])
ipc_raw.columns = ["FECHA", "IPC"]
ipc_raw = ipc_raw.dropna(subset=["FECHA"])
ipc_raw["FECHA"] = pd.to_datetime(ipc_raw["FECHA"], errors="coerce")
ipc = ipc_raw.dropna().set_index("FECHA")["IPC"].astype(float)
base_ipc = ipc[ipc.index.year == IPC_BASE_PROM_ANUAL]
IPC_BASE = float(base_ipc.mean()) if len(base_ipc) == 12 else IPC_BASE_FIJO
deflactor = (ipc / IPC_BASE)              # igual a la fila "deflactor" del workbook manual

# Universo de fechas: desde AÑO_INICIO hasta el último mes con algún devengado
fechas_con_datos = pd.concat([df["FECHA"] for df in inputs.values() if not df.empty])
ULTIMO_MES = fechas_con_datos.max()
AÑOS = list(range(AÑO_INICIO, ULTIMO_MES.year + 1))
FECHAS = [datetime(a, m, 1) for a in AÑOS for m in range(1, 13)]

print(f"Series: {AÑO_INICIO}-01 a {ULTIMO_MES:%Y-%m} · IPC base {IPC_BASE_PROM_ANUAL} = {IPC_BASE:.2f} · último IPC {ipc.index.max():%Y-%m}")
if faltantes:
    print("⚠️ Sin input (los bloques quedarán vacíos):", ", ".join(faltantes))
    if any(x.startswith("DEV Deuda") for x in faltantes):
        print("   (sin DEV Deuda el subtotal SERVICIO DE LA DEUDA igual se llena con la genérica 2.8 de DEV Generica; "
              "solo quedan vacías Intereses+comisiones y Amortización en los meses con deuda)")
if not controles.empty:
    print("Controles leídos:", len(controles), "filas ·", (controles["ESTADO"] != "OK").sum(), "con estado distinto de OK")


In [ ]:
# ═══ BLOQUE 4 · CONSTRUCCIÓN DEL WORKBOOK ═══
F_TIT = Font(name="Arial", size=9, bold=True); F_NOR = Font(name="Arial", size=9)
F_SEC = Font(name="Arial", size=9, bold=True); F_SUB = Font(name="Arial", size=9, italic=True)
FILL_H = PatternFill("solid", fgColor="D9E1F2"); FILL_T = PatternFill("solid", fgColor="F2F2F2")
THIN = Side(style="thin", color="999999")

def serie_mensual(df, filtro_cols, valor="DEVENGADO"):
    """-> dict {FECHA: suma} de un DataFrame ya filtrado por departamento."""
    if df is None or df.empty:
        return {}
    for col, vals in filtro_cols.items():
        if vals is not None:
            df = df[df[col].isin(vals)]
    return df.groupby("FECHA")[valor].sum().to_dict()

class Hoja:
    """Escribe una hoja con el formato del workbook manual: fila 4 años, fila 5 fechas,
    columna 'Total' tras cada diciembre, y al final Var.% acumulada Ene.-Mes."""
    def __init__(self, wb, nombre, titulo, cuadro, unidad="(Miles de soles)"):
        self.ws = wb.create_sheet(nombre)
        self.ws["A1"] = cuadro; self.ws["A1"].font = F_TIT
        self.ws["A2"] = titulo; self.ws["A2"].font = F_TIT
        self.ws["A3"] = unidad; self.ws["A3"].font = F_SUB
        self.col_de = {}                 # FECHA -> columna
        self.cols_total = {}             # AÑO -> columna Total
        c = 2
        for a in AÑOS:
            self.ws.cell(4, c, a).font = F_TIT
            for m in range(1, 13):
                f = datetime(a, m, 1)
                cell = self.ws.cell(5, c, f); cell.number_format = "mmm-yy"; cell.font = F_NOR
                cell.alignment = Alignment(horizontal="center")
                self.col_de[f] = c; c += 1
            self.ws.cell(4, c, "Total").font = F_TIT
            self.ws.cell(5, c, a).font = F_TIT
            self.cols_total[a] = c; c += 1
        self.c_fin = c - 1                       # última columna de datos
        self.c_var = c + 1                       # columnas de variación
        self.fila = 7
        self.ws.column_dimensions["A"].width = 46
        for cc in range(2, self.c_var + 6):
            self.ws.column_dimensions[L(cc)].width = 9
        self.ws.freeze_panes = "B6"
        # encabezados del bloque de variación
        u = ULTIMO_MES
        self.ws.cell(4, self.c_var, f"Var.% {u.year}/{u.year-1}").font = F_TIT
        self.ws.cell(5, self.c_var, u.strftime("%b-%y")).font = F_NOR
        self.ws.cell(4, self.c_var + 1, "Var.% acum.").font = F_TIT
        self.ws.cell(5, self.c_var + 1, f"Ene.-{u.strftime('%b.')}").font = F_NOR
        self.ws.cell(4, self.c_var + 2, u.year - 1).font = F_TIT
        self.ws.cell(5, self.c_var + 2, f"Ene.-{u.strftime('%b.')}").font = F_NOR
        self.ws.cell(4, self.c_var + 3, u.year).font = F_TIT
        self.ws.cell(5, self.c_var + 3, f"Ene.-{u.strftime('%b.')}").font = F_NOR
        self.ws.cell(4, self.c_var + 4, "Dif. acum.").font = F_TIT

    def _variacion(self, r):
        """Fórmulas de variación para la fila r (mismo criterio que el workbook manual)."""
        u = ULTIMO_MES
        c_act, c_ant = self.col_de.get(u), self.col_de.get(datetime(u.year - 1, u.month, 1))
        ws = self.ws
        if c_ant:
            ws.cell(r, self.c_var, f'=IFERROR(({L(c_act)}{r}/{L(c_ant)}{r}-1)*100,"")')
        a0, a1 = self.col_de.get(datetime(u.year - 1, 1, 1)), self.col_de.get(datetime(u.year, 1, 1))
        if a0:
            ws.cell(r, self.c_var + 2, f"=SUM({L(a0)}{r}:{L(c_ant)}{r})")
        ws.cell(r, self.c_var + 3, f"=SUM({L(a1)}{r}:{L(c_act)}{r})")
        if a0:
            ws.cell(r, self.c_var + 1, f'=IFERROR(({L(self.c_var+3)}{r}/{L(self.c_var+2)}{r}-1)*100,"")')
            ws.cell(r, self.c_var + 4, f"={L(self.c_var+3)}{r}-{L(self.c_var+2)}{r}")
        for cc in range(self.c_var, self.c_var + 5):
            ws.cell(r, cc).number_format = "#,##0.0"; ws.cell(r, cc).font = F_NOR

    def _totales(self, r):
        for a, c in self.cols_total.items():
            ini, fin = self.col_de[datetime(a, 1, 1)], self.col_de[datetime(a, 12, 1)]
            cell = self.ws.cell(r, c, f"=SUM({L(ini)}{r}:{L(fin)}{r})")
            cell.number_format = "#,##0"; cell.font = F_TIT; cell.fill = FILL_T

    def fila_datos(self, etiqueta, valores, escala=1 / 1000, fmt="#,##0", font=F_NOR, variacion=True, totales=True):
        """valores: dict {FECHA: monto en soles}. Escribe en miles; meses sin dato quedan vacíos."""
        r = self.fila
        self.ws.cell(r, 1, etiqueta).font = font
        for f, c in self.col_de.items():
            if f in valores and f <= ULTIMO_MES:
                cell = self.ws.cell(r, c, round(valores[f] * escala, 3 if escala != 1 / 1000 else 0))
                cell.number_format = fmt; cell.font = font
        if totales:
            self._totales(r)
        if variacion:
            self._variacion(r)
        self.fila += 1
        return r

    def fila_formula(self, etiqueta, plantilla, font=F_SEC, fmt="#,##0", variacion=True, totales=True):
        """plantilla usa {c} como letra de columna, p.ej. '=SUM({c}9:{c}13)'."""
        r = self.fila
        self.ws.cell(r, 1, etiqueta).font = font
        for f, c in self.col_de.items():
            cell = self.ws.cell(r, c, plantilla.format(c=L(c), r=r)); cell.number_format = fmt; cell.font = font
        if totales:
            self._totales(r)
        if variacion:
            self._variacion(r)
        self.fila += 1
        return r

    def fila_ratio(self, etiqueta, fila_num, fila_den, factor=1, font=F_NOR, fmt="#,##0"):
        return self.fila_formula(etiqueta, f"=IFERROR({{c}}{fila_num}/{{c}}{fila_den}{'*' + str(factor) if factor != 1 else ''},\"\")",
                                 font=font, fmt=fmt)

    def texto(self, t, font=F_SUB):
        self.ws.cell(self.fila, 1, t).font = font; self.fila += 1

    def salto(self, n=1):
        self.fila += n

def hoja_gasto(wb, n, cod_depto, nombre_depto, cuadro):
    gen = inputs[(n["corto"], "Generica")]; deu = inputs[(n["corto"], "Deuda")]; fue = inputs[(n["corto"], "Fuente")]
    gen = gen[gen["COD_DEPTO"] == cod_depto] if not gen.empty else gen
    deu = deu[deu["COD_DEPTO"] == cod_depto] if not deu.empty else deu
    fue = fue[fue["COD_DEPTO"] == cod_depto] if not fue.empty else fue
    if not gen.empty:
        gen = gen.assign(CLASE=[clasificar_generica(c, e) for c, e in zip(gen["COD_GENERICA"], gen["GENERICA"])])
    if not fue.empty:
        fue = fue.assign(CLASE=fue["FUENTE"].map(clasificar_fuente))
    # ── CORRECCIÓN servicio de la deuda ──
    # 1) el subtotal SERVICIO DE LA DEUDA sale de la genérica 2.8 (tipo de transacción 7) de DEV Generica, que siempre se
    #    raspa; antes era la suma de Intereses + Amortización y quedaba en 0 cuando el desglose (DEV Deuda) fallaba o faltaba.
    #    Respaldo: la fila TOTAL que ahora guarda el scraper en DEV Deuda.
    # 2) las filas de DEV Deuda con CLASIFICACION = OTRO se reclasifican por el texto de la RUTA.
    # 3) en los meses con dato de genéricas y sin deuda, la deuda se escribe como 0 (no queda en blanco).
    if not deu.empty:
        deu = deu.assign(CLASIFICACION=[c if c in ("TOTAL", "AMORTIZACION", "INTERESES Y COMISIONES") else clasificar_deuda(r)
                                        for c, r in zip(deu["CLASIFICACION"].astype(str), deu["RUTA"].astype(str))])
    tot_deuda = {}
    if not gen.empty:
        es_deuda = gen["COD_GENERICA"].astype(str).str.strip().str.split("-").str[0].eq("7") | gen["GENERICA"].map(lambda e: "servicio de la deuda" in normalizar(e))
        tot_deuda = {f: 0.0 for f in gen["FECHA"].unique()}
        tot_deuda.update(gen[es_deuda].groupby("FECHA")["DEVENGADO"].sum().to_dict())
    elif not deu.empty:
        tot_deuda = serie_mensual(deu, {"CLASIFICACION": ["TOTAL"]})
    h = Hoja(wb, n["hoja_g"], f"{nombre_depto}: GASTOS NO FINANCIEROS DEL {n['nombre']}   1/", cuadro)
    filas = {}
    r_total = h.fila; h.fila += 1                                      # TOTAL DE GASTOS (se llena al final)
    secciones = {}
    seccion = None
    for etiqueta, cods in GENERICAS:
        if cods is None:
            seccion = etiqueta; secciones[seccion] = [h.fila, None]; h.fila += 1
            continue
        if isinstance(cods, str) and cods.startswith("DEUDA:"):
            vals = serie_mensual(deu, {"CLASIFICACION": [cods.split(":", 1)[1]]})
            vals.update({f: 0.0 for f, t in tot_deuda.items() if t == 0 and f not in vals})
        else:
            vals = serie_mensual(gen, {"CLASE": [etiqueta]})
        filas[etiqueta] = h.fila_datos(etiqueta, vals)
        secciones[seccion][1] = filas[etiqueta]
    # subtotales y total (fórmulas, como en el manual)
    for sec, (r, r_fin) in secciones.items():
        r_ini = r + 1
        h.fila, guardado = r, h.fila
        if sec == "SERVICIO DE LA DEUDA" and tot_deuda:
            h.fila_datos(sec, tot_deuda, font=F_SEC)          # total de la genérica 2.8 (ver CORRECCIÓN arriba)
        else:
            h.fila_formula(sec, f"=SUM({{c}}{r_ini}:{{c}}{r_fin})")
        h.fila = guardado
    h.fila, guardado = r_total, h.fila
    h.fila_formula("TOTAL DE GASTOS", "=" + "+".join(f"{{c}}{r}" for r, _ in secciones.values()), font=F_TIT)
    h.fila = guardado
    h.salto()
    r_fuente = h.fila; h.fila += 1
    r_f = []
    for etiqueta, _ in FUENTES:
        r_f.append(h.fila_datos(etiqueta, serie_mensual(fue, {"CLASE": [etiqueta]})))
    h.fila, guardado = r_fuente, h.fila
    h.fila_formula("GASTOS POR FUENTE", f"=SUM({{c}}{r_f[0]}:{{c}}{r_f[-1]})")
    h.fila = guardado
    h.fila_formula("   Diferencia total - fuentes (control)", f"={{c}}{r_total}-{{c}}{r_fuente}", font=F_SUB, variacion=False, totales=False)
    r_deu = secciones["SERVICIO DE LA DEUDA"][0]
    h.fila_formula("   Deuda sin desglosar: total - (intereses + amortización) (control)",
                   f"={{c}}{r_deu}-{{c}}{filas['Intereses+comisiones']}-{{c}}{filas['Amortización']}", font=F_SUB, variacion=False, totales=False)
    h.texto("Fuente: MEF, Consulta Amigable (scraper DEV Generica / DEV Fuente / DEV Deuda)."); h.texto("1/ Preliminar")
    h.salto()
    # ── bloque real ──
    h.texto(f"{nombre_depto}: GASTOS DEL {n['nombre']} EN TÉRMINOS REALES (deflactados con IPC Lima, base {IPC_BASE_PROM_ANUAL} = 100)", F_TIT)
    r_defl = h.fila_datos("Deflactor (IPC / IPC base)", {f: v for f, v in deflactor.items()}, escala=1, fmt="0.0000", variacion=False, totales=False)
    h.fila_ratio("TOTAL DE GASTOS", r_total, r_defl, font=F_TIT)
    for sec, (r, _) in secciones.items():
        h.fila_ratio("  " + sec, r, r_defl, font=F_SEC)
        for etiqueta, cods in GENERICAS:
            if etiqueta in filas and _seccion_de(etiqueta) == sec:
                h.fila_ratio("    " + etiqueta, filas[etiqueta], r_defl)
    h.fila_ratio("GASTOS POR FUENTE DE FINANCIAMIENTO", r_fuente, r_defl, font=F_SEC)
    for (etiqueta, _), r in zip(FUENTES, r_f):
        h.fila_ratio("    " + etiqueta, r, r_defl)
    return h

def _seccion_de(etiqueta):
    sec = None
    for e, c in GENERICAS:
        if c is None:
            sec = e
        if e == etiqueta:
            return sec

def hoja_inversion(wb, n, cod_depto, nombre_depto, cuadro):
    fun = inputs[(n["corto"], "Funcion")]; inv = inputs[(n["corto"], "InvFuente")]
    fun = fun[fun["COD_DEPTO"] == cod_depto] if not fun.empty else fun
    inv = inv[inv["COD_DEPTO"] == cod_depto] if not inv.empty else inv
    if not inv.empty:
        inv = inv.assign(CLASE=inv["FUENTE"].map(clasificar_fuente))
    h = Hoja(wb, n["hoja_i"], f"{nombre_depto}: INVERSIONES NO FINANCIERAS DEL {n['nombre']}   1/", cuadro)
    r_tot = h.fila; h.fila += 2
    r_ini = h.fila
    for f in FUNCIONES:
        cods = [f[:2]]
        if AGRUPAR_COMUNICACIONES_EN_TRANSPORTE:
            if f[:2] == "15":
                cods = ["15", "16"]
            elif f[:2] == "16":
                cods = []
        h.fila_datos(f + (" (incl. en Transporte)" if cods == [] else ""), serie_mensual(fun, {"COD_FUNCION": cods}))
    r_fin = h.fila - 1
    h.fila, g = r_tot, h.fila
    h.fila_formula("INVERSION POR FUNCIONES", f"=SUM({{c}}{r_ini}:{{c}}{r_fin})", font=F_TIT)
    h.fila = g; h.salto()
    r_tf = h.fila; h.fila += 2
    r_fi = h.fila
    for etiqueta, _ in FUENTES:
        h.fila_datos(etiqueta, serie_mensual(inv, {"CLASE": [etiqueta]}))
    r_ff = h.fila - 1
    h.fila, g = r_tf, h.fila
    h.fila_formula("INVERSION POR FUENTE", f"=SUM({{c}}{r_fi}:{{c}}{r_ff})", font=F_TIT)
    h.fila = g
    h.fila_formula("   Diferencia funciones - fuentes (control)", f"={{c}}{r_tot}-{{c}}{r_tf}", font=F_SUB, variacion=False, totales=False)
    h.texto("Fuente: MEF, Consulta Amigable (scraper DEV Funcion / DEV InvFuente). Genérica 2.6 Adquisición de activos no financieros."); h.texto("1/ Preliminar")
    h.salto()
    h.texto(f"{nombre_depto}: INVERSIÓN DEL {n['nombre']} EN TÉRMINOS REALES (IPC Lima, base {IPC_BASE_PROM_ANUAL} = 100)", F_TIT)
    r_defl = h.fila_datos("Deflactor (IPC / IPC base)", {f: v for f, v in deflactor.items()}, escala=1, fmt="0.0000", variacion=False, totales=False)
    h.fila_ratio("INVERSION POR FUNCIONES (real)", r_tot, r_defl, font=F_TIT)
    for i, f in enumerate(FUNCIONES):
        h.fila_ratio("    " + f, r_ini + i, r_defl)
    return h

def hoja_pim(wb, n, cod_depto, nombre_depto):
    """PIM anual por función (todas las genéricas de 2.6 según el scraper PIM) vs devengado 2.6 acumulado."""
    ws = wb.create_sheet(n["hoja_p"])
    ws["A1"] = f"{nombre_depto}: PIM Y AVANCE DE LA INVERSIÓN DEL {n['nombre']} POR FUNCIÓN"; ws["A1"].font = F_TIT
    ws["A2"] = "(Soles; devengado acumulado de la genérica 2.6 al último mes disponible de cada año)"; ws["A2"].font = F_SUB
    ws.column_dimensions["A"].width = 46
    fun = inputs[(n["corto"], "Funcion")]
    fun = fun[fun["COD_DEPTO"] == cod_depto] if not fun.empty else fun
    p = pim[(pim["COD_DEPTO"] == cod_depto) & (pim["NIVEL DE GOBIERNO"].str.upper() == n["pim"])] if not pim.empty else pim
    c = 2
    ws.cell(4, 1, "FUNCIÓN").font = F_TIT
    for a in AÑOS:
        ult = fun[fun["AÑO"] == a]["FECHA"].max() if not fun.empty and (fun["AÑO"] == a).any() else None
        etiqueta_mes = ult.strftime("%b-%y") if ult is not None and pd.notna(ult) else "s/d"
        for j, t in enumerate([f"PIM {a}", f"Devengado a {etiqueta_mes}", "Avance %"]):
            cell = ws.cell(4, c + j, t); cell.font = F_TIT; cell.fill = FILL_H; cell.alignment = Alignment(horizontal="center", wrap_text=True)
            ws.column_dimensions[L(c + j)].width = 14
        for i, f in enumerate(FUNCIONES):
            r = 6 + i
            ws.cell(r, 1, f).font = F_NOR
            v_pim = float(p[p["COD_FUNCION"] == f[:2]]["PIM"].sum()) if not p.empty and (p["AÑO"] == a).any() else None
            if not p.empty:
                v_pim = float(p[(p["COD_FUNCION"] == f[:2]) & (p["AÑO"] == a)]["PIM"].sum())
            v_dev = float(fun[(fun["COD_FUNCION"] == f[:2]) & (fun["AÑO"] == a)]["DEVENGADO"].sum()) if not fun.empty else None
            if v_pim is not None:
                ws.cell(r, c, v_pim).number_format = "#,##0"
            if v_dev is not None:
                ws.cell(r, c + 1, v_dev).number_format = "#,##0"
            ws.cell(r, c + 2, f'=IFERROR({L(c+1)}{r}/{L(c)}{r}*100,"")').number_format = "0.0"
        r = 6 + len(FUNCIONES)
        ws.cell(r, 1, "TOTAL").font = F_TIT
        ws.cell(r, c, f"=SUM({L(c)}6:{L(c)}{r-1})").number_format = "#,##0"
        ws.cell(r, c + 1, f"=SUM({L(c+1)}6:{L(c+1)}{r-1})").number_format = "#,##0"
        ws.cell(r, c + 2, f'=IFERROR({L(c+1)}{r}/{L(c)}{r}*100,"")').number_format = "0.0"
        for cc in range(c, c + 3):
            ws.cell(r, cc).font = F_TIT; ws.cell(r, cc).fill = FILL_T
        c += 4
    ws.cell(r + 2, 1, "Fuente: MEF, Consulta Amigable (scrapers PIM Funcion y DEV Funcion). PIM = presupuesto institucional modificado vigente a la fecha de extracción.").font = F_SUB
    ws.freeze_panes = "B5"

def hoja_control(wb, cod_depto, nombre_depto):
    ws = wb.create_sheet("CONTROL")
    ws["A1"] = f"CONTROLES DE LOS SCRAPERS · {nombre_depto}"; ws["A1"].font = F_TIT
    ws["A2"] = "Resumen de los CSV de control (suma de meses vs vista anual del MEF; PIM por nivel vs total). ESTADO distinto de OK = revisar antes de usar."; ws["A2"].font = F_SUB
    ws["A4"] = f"Generado: {date.today().isoformat()} · último mes con datos: {ULTIMO_MES:%Y-%m}"; ws["A4"].font = F_NOR
    ws.column_dimensions["A"].width = 70; ws.column_dimensions["B"].width = 8
    for j, t in enumerate(["Archivo de control", "Año", "Filas", "OK", "No OK", "Estados", "Extracción"]):
        cell = ws.cell(6, 1 + j, t); cell.font = F_TIT; cell.fill = FILL_H
    if controles.empty:
        ws.cell(7, 1, "No se encontraron CSV de control.").font = F_SUB
        return
    sub = controles[controles["DEPARTAMENTO"].astype(str).str.startswith(cod_depto)]
    r = 7
    for archivo, g in sub.groupby("ARCHIVO"):
        ok = (g["ESTADO"] == "OK").sum()
        ws.cell(r, 1, archivo).font = F_NOR
        ws.cell(r, 2, int(g["AÑO"].iloc[0]) if pd.notna(g["AÑO"].iloc[0]) else "")
        ws.cell(r, 3, len(g)); ws.cell(r, 4, int(ok)); ws.cell(r, 5, int(len(g) - ok))
        ws.cell(r, 6, ", ".join(f"{k}: {v}" for k, v in g["ESTADO"].value_counts().items()))
        ws.cell(r, 7, str(g["FECHA_EXTRACCION"].iloc[0]))
        if ok < len(g):
            for cc in range(1, 8):
                ws.cell(r, cc).font = Font(name="Arial", size=9, bold=True, color="C00000")
        r += 1
    for cc, w in zip("CDEFG", [7, 7, 7, 40, 12]):
        ws.column_dimensions[cc].width = w


In [ ]:
# ═══ BLOQUE 5 · GENERAR UN WORKBOOK POR DEPARTAMENTO ═══
generados = []
for cod, (nombre, archivo) in DEPARTAMENTOS.items():
    wb = Workbook(); wb.remove(wb.active)
    cuadros = iter(range(18, 30))
    for n in NIVELES:
        hoja_gasto(wb, n, cod, nombre, f"CUADRO Nº {next(cuadros)}")
        hoja_inversion(wb, n, cod, nombre, f"CUADRO Nº {next(cuadros)}")
    for n in NIVELES:
        hoja_pim(wb, n, cod, nombre)
    hoja_control(wb, cod, nombre)
    salida = os.path.join(RUTA_SALIDA, f"Fiscal-{archivo}{SUFIJO_SALIDA}.xlsx")
    wb.save(salida); generados.append(salida)
    print("💾", salida)

print("\nResumen de inputs usados (filas por nivel y dimensión):")
display(pd.DataFrame([{"nivel": k[0], "dimensión": k[1], "filas": len(v), "desde": (v["FECHA"].min().strftime("%Y-%m") if not v.empty else "-"),
                       "hasta": (v["FECHA"].max().strftime("%Y-%m") if not v.empty else "-")} for k, v in inputs.items()]))
if faltantes:
    print("⚠️ Bloques vacíos por falta de input:", ", ".join(faltantes))


In [ ]:
# ═══ BLOQUE 6 · VARIACIONES ALARMANTES (inversión 2.6 en soles corrientes) ═══
# Total de inversión de cada nivel de gobierno (y de los tres juntos) frente al periodo anterior, y ranking de los
# VARIACIONES_TOP mayores aumentos y mayores caídas del devengado de inversión, en SOLES (no en %), por departamento
# y nivel de gobierno (funciones; en G.L. también municipalidades) y entre todos los niveles a la vez.
# Se compara el último mes con datos contra el mismo mes del año anterior y el acumulado Ene.-Mes contra el del año
# anterior (opcional: contra el mes anterior). Un nivel sin datos en alguno de los meses comparados se omite (no se asume 0).
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Set", "Oct", "Nov", "Dic"]
CORTO_NIVEL = {"GOBIERNO NACIONAL": "G.N.", "GOBIERNO REGIONAL": "G.R.", "GOBIERNOS LOCALES": "G.L."}

def etq_mes(f):
    return f"{MESES_ES[f.month - 1]}-{f:%y}"

def etq_periodo(fechas):
    return etq_mes(fechas[0]) if len(fechas) == 1 else f"{etq_mes(fechas[0])} a {etq_mes(fechas[-1])}"

def nombre_funcion(cod):
    if AGRUPAR_COMUNICACIONES_EN_TRANSPORTE and cod in ("15", "16"):
        return "15: TRANSPORTE (incl. 16: Comunicaciones)"
    return next((f for f in FUNCIONES if f[:2] == cod), cod)

# Serie larga: DEPARTAMENTO x NIVEL x TIPO x ITEM x FECHA -> DEVENGADO (soles)
partes, disponibles = [], {}
for n in NIVELES:
    f = inputs[(n["corto"], "Funcion")]
    if f.empty:
        continue
    disponibles[n["nombre"]] = set(pd.to_datetime(f["FECHA"]))
    partes.append(pd.DataFrame({"COD_DEPTO": f["COD_DEPTO"], "NIVEL": n["nombre"], "TIPO": "Función",
                                "ITEM": f["COD_FUNCION"].map(nombre_funcion), "FECHA": pd.to_datetime(f["FECHA"]),
                                "DEVENGADO": f["DEVENGADO"]}))
if not detalle_municipal.empty:
    m = detalle_municipal
    partes.append(pd.DataFrame({"COD_DEPTO": m["COD_DEPTO"], "NIVEL": "GOBIERNOS LOCALES", "TIPO": "Municipalidad",
                                "ITEM": m["MUNICIPALIDAD"].astype(str), "FECHA": pd.to_datetime(m["FECHA"]),
                                "DEVENGADO": m["DEVENGADO"]}))
serie = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()

u = pd.Timestamp(ULTIMO_MES)
u_ant, m_ant = u - pd.DateOffset(years=1), u - pd.DateOffset(months=1)
DEF_COMPARACIONES = {
    "MES_AÑO_ANTERIOR": ("Mes vs mismo mes del año anterior", [u_ant], [u]),
    "MES_ANTERIOR":     ("Mes vs mes anterior", [m_ant], [u]),
    "ACUMULADO":        (f"Acumulado Ene.-{MESES_ES[u.month - 1]} vs año anterior",
                         list(pd.date_range(f"{u_ant.year}-01-01", u_ant, freq="MS")),
                         list(pd.date_range(f"{u.year}-01-01", u, freq="MS"))),
}
comparaciones = [DEF_COMPARACIONES[k] for k in VARIACIONES_COMPARACIONES]

# Bloques del ranking: (título, filtro de la serie, nombre del ítem)
BLOQUES_VAR = [(f"{n['nombre']} · por función", n["nombre"], "Función") for n in NIVELES]
BLOQUES_VAR.insert(0, ("TODOS LOS NIVELES · nivel y función", None, "Función"))
BLOQUES_VAR.append(("GOBIERNOS LOCALES · por municipalidad", "GOBIERNOS LOCALES", "Municipalidad"))

notas_var = []

def var_pct(anterior, actual):
    return (actual / anterior - 1) * 100 if anterior else np.nan

def tabla_variacion(cod_depto, bloque, comp):
    """-> DataFrame ITEM | ANTERIOR | ACTUAL | VARIACION | VAR_PCT con todos los ítems del bloque (o None si no hay datos)."""
    titulo, nivel, tipo = bloque
    nombre_comp, f_a, f_b = comp
    niveles = [nivel] if nivel else list(disponibles)
    ok = [nv for nv in niveles if nv in disponibles and set(f_a + f_b) <= disponibles[nv]]
    for nv in sorted(set(niveles) - set(ok)):
        nota = f"{nombre_comp}: sin datos de {nv} para {etq_periodo(f_a)} o {etq_periodo(f_b)} (no se compara)."
        if nota not in notas_var:
            notas_var.append(nota)
    if not ok or serie.empty:
        return None
    s = serie[(serie["COD_DEPTO"] == cod_depto) & (serie["TIPO"] == tipo) & (serie["NIVEL"].isin(ok))]
    if nivel is None:
        s = s.assign(ITEM=s["NIVEL"].map(CORTO_NIVEL) + " · " + s["ITEM"])
    a = s[s["FECHA"].isin(f_a)].groupby("ITEM")["DEVENGADO"].sum().rename("ANTERIOR")
    b = s[s["FECHA"].isin(f_b)].groupby("ITEM")["DEVENGADO"].sum().rename("ACTUAL")
    t = pd.concat([a, b], axis=1).fillna(0.0).reset_index()
    t["VARIACION"] = t["ACTUAL"] - t["ANTERIOR"]
    t["VAR_PCT"] = [var_pct(x, y) for x, y in zip(t["ANTERIOR"], t["ACTUAL"])]
    return t

def tabla_totales(cod_depto, comp):
    """Total mensual (o acumulado) de inversión de cada nivel de gobierno y de los tres juntos, como la vista
    Departamento (Meta) > Nivel de Gobierno del MEF. -> lista de (nivel, anterior, actual); None si el nivel no tiene datos."""
    nombre_comp, f_a, f_b = comp
    filas = []
    for n in NIVELES:
        nv = n["nombre"]
        if nv not in disponibles or not set(f_a + f_b) <= disponibles[nv]:
            filas.append((nv, None, None))
            continue
        s = serie[(serie["COD_DEPTO"] == cod_depto) & (serie["TIPO"] == "Función") & (serie["NIVEL"] == nv)]
        filas.append((nv, s[s["FECHA"].isin(f_a)]["DEVENGADO"].sum(), s[s["FECHA"].isin(f_b)]["DEVENGADO"].sum()))
    completos = all(a is not None for _, a, _ in filas)
    filas.append(("TOTAL TRES NIVELES", sum(a for _, a, _ in filas) if completos else None,
                  sum(b for _, _, b in filas) if completos else None))
    return filas

def tops(t):
    sube = t[t["VARIACION"] > 0].nlargest(VARIACIONES_TOP, "VARIACION")
    baja = t[t["VARIACION"] < 0].nsmallest(VARIACIONES_TOP, "VARIACION")
    return sube, baja

# ── Workbook ──
F_SUBE = Font(name="Arial", size=9, color="1E7B34"); F_BAJA = Font(name="Arial", size=9, color="C00000")
F_SUBE_B = Font(name="Arial", size=9, bold=True, color="1E7B34"); F_BAJA_B = Font(name="Arial", size=9, bold=True, color="C00000")
FILL_B = PatternFill("solid", fgColor="FCE4D6")
ANCHO_COMP = 6                                   # # | Ítem | Anterior | Actual | Diferencia S/ | Var. %
TITULO_TOTALES = "TOTAL POR NIVEL DE GOBIERNO"

def encabezado_bloque(ws, r, titulo):
    ws.cell(r, 1, titulo).font = F_TIT
    for cc in range(1, len(comparaciones) * (ANCHO_COMP + 1)):
        ws.cell(r, cc).fill = FILL_H

def encabezado_comparacion(ws, r, c0, comp, primera="Ítem"):
    nombre_comp, f_a, f_b = comp
    ws.cell(r, c0, f"{nombre_comp}: {etq_periodo(f_b)} vs {etq_periodo(f_a)}").font = F_TIT
    for j, t in enumerate(["#", primera, etq_periodo(f_a) if len(f_a) == 1 else f"Ene.-{MESES_ES[u.month-1]} {u_ant.year}",
                           etq_periodo(f_b) if len(f_b) == 1 else f"Ene.-{MESES_ES[u.month-1]} {u.year}", "Diferencia S/", "Var. %"]):
        cell = ws.cell(r + 1, c0 + j, t); cell.font = F_TIT; cell.fill = FILL_T
        cell.alignment = Alignment(horizontal="center" if j != 1 else "left")

wb = Workbook(); ws_res = wb.active; ws_res.title = "RESUMEN"
detalle = []
for cod, (nombre, _) in DEPARTAMENTOS.items():
    ws = wb.create_sheet(nombre[:31])
    ws["A1"] = f"{nombre}: VARIACIONES ALARMANTES DE LA INVERSIÓN (GENÉRICA 2.6) POR NIVEL DE GOBIERNO"; ws["A1"].font = F_TIT
    ws["A2"] = (f"Soles corrientes · {VARIACIONES_TOP} mayores aumentos y {VARIACIONES_TOP} mayores caídas en monto (no en %) · "
                f"último mes con datos: {etq_mes(u)}"); ws["A2"].font = F_SUB
    for k in range(len(comparaciones)):
        c0 = 1 + k * (ANCHO_COMP + 1)
        for j, w in enumerate([4, 46, 14, 14, 14, 8]):
            ws.column_dimensions[L(c0 + j)].width = w
        ws.column_dimensions[L(c0 + ANCHO_COMP)].width = 3
    # ── Totales de inversión por nivel de gobierno (y de los tres juntos) vs el periodo anterior ──
    r = 4
    encabezado_bloque(ws, r, TITULO_TOTALES)
    r += 1
    for k, comp in enumerate(comparaciones):
        c0 = 1 + k * (ANCHO_COMP + 1)
        encabezado_comparacion(ws, r, c0, comp, primera="Nivel de gobierno")
        for i, (nv, a, b) in enumerate(tabla_totales(cod, comp)):
            total = nv == "TOTAL TRES NIVELES"
            ws.cell(r + 2 + i, c0 + 1, nv).font = F_TIT if total else F_NOR
            if a is None:
                ws.cell(r + 2 + i, c0 + 2, "Sin datos para comparar (ver RESUMEN)").font = F_SUB
                continue
            dif, pct = b - a, var_pct(a, b)
            for j, v in enumerate([a, b, dif, pct if pd.notna(pct) else "n.a."], start=2):
                cell = ws.cell(r + 2 + i, c0 + j, v); cell.number_format = "0.0" if j == 5 else "#,##0"
                cell.font = (F_SUBE_B if dif > 0 else F_BAJA_B if dif < 0 else F_TIT) if j == 4 and total else \
                            (F_SUBE if dif > 0 else F_BAJA if dif < 0 else F_NOR) if j == 4 else (F_TIT if total else F_NOR)
                if total:
                    cell.fill = FILL_T
            detalle.append({"DEPARTAMENTO": nombre, "BLOQUE": TITULO_TOTALES, "COMPARACIÓN": comp[0],
                            "PERIODO ANTERIOR": etq_periodo(comp[1]), "PERIODO ACTUAL": etq_periodo(comp[2]), "ÍTEM": nv,
                            "ANTERIOR S/": a, "ACTUAL S/": b, "DIFERENCIA S/": dif, "VAR. %": pct, "RANKING": ""})
    r += 2 + len(NIVELES) + 1 + 1
    for bloque in BLOQUES_VAR:
        encabezado_bloque(ws, r, bloque[0])
        r += 1
        resultados = []
        for k, comp in enumerate(comparaciones):
            c0 = 1 + k * (ANCHO_COMP + 1)
            nombre_comp, f_a, f_b = comp
            encabezado_comparacion(ws, r, c0, comp)
            t = tabla_variacion(cod, bloque, comp)
            resultados.append(t)
            if t is not None:
                sube, baja = tops(t)
                for _, x in t.iterrows():
                    puesto = (f"Aumento #{list(sube['ITEM']).index(x['ITEM']) + 1}" if x["ITEM"] in set(sube["ITEM"]) else
                              f"Caída #{list(baja['ITEM']).index(x['ITEM']) + 1}" if x["ITEM"] in set(baja["ITEM"]) else "")
                    detalle.append({"DEPARTAMENTO": nombre, "BLOQUE": bloque[0], "COMPARACIÓN": nombre_comp,
                                    "PERIODO ANTERIOR": etq_periodo(f_a), "PERIODO ACTUAL": etq_periodo(f_b), "ÍTEM": x["ITEM"],
                                    "ANTERIOR S/": x["ANTERIOR"], "ACTUAL S/": x["ACTUAL"], "DIFERENCIA S/": x["VARIACION"],
                                    "VAR. %": x["VAR_PCT"], "RANKING": puesto})
        r += 2
        for etiqueta, idx, f_tit, f_fila in [("▲ Mayores aumentos", 0, F_SUBE_B, F_SUBE), ("▼ Mayores caídas", 1, F_BAJA_B, F_BAJA)]:
            for k in range(len(comparaciones)):
                ws.cell(r, 2 + k * (ANCHO_COMP + 1), etiqueta).font = f_tit
            r += 1
            for k, t in enumerate(resultados):
                c0 = 1 + k * (ANCHO_COMP + 1)
                if t is None:
                    ws.cell(r, c0 + 1, "Sin datos para comparar (ver RESUMEN)").font = F_SUB
                    continue
                filas = tops(t)[idx]
                if filas.empty:
                    ws.cell(r, c0 + 1, "Ninguno").font = F_SUB
                for i, (_, x) in enumerate(filas.iterrows()):
                    vals = [i + 1, x["ITEM"], x["ANTERIOR"], x["ACTUAL"], x["VARIACION"], x["VAR_PCT"] if pd.notna(x["VAR_PCT"]) else "n.a."]
                    for j, v in enumerate(vals):
                        cell = ws.cell(r + i, c0 + j, v); cell.font = f_fila if j == 4 else F_NOR
                        cell.number_format = "0.0" if j == 5 else "#,##0"
            r += VARIACIONES_TOP + 1
        r += 1
    ws.freeze_panes = "A4"

# Hoja DETALLE: todas las variaciones (no solo el top), para filtrar
ws_det = wb.create_sheet("DETALLE")
det = pd.DataFrame(detalle)
cols_det = ["DEPARTAMENTO", "BLOQUE", "COMPARACIÓN", "PERIODO ANTERIOR", "PERIODO ACTUAL", "ÍTEM",
            "ANTERIOR S/", "ACTUAL S/", "DIFERENCIA S/", "VAR. %", "RANKING"]
for j, (t, w) in enumerate(zip(cols_det, [12, 38, 34, 18, 18, 50, 14, 14, 14, 8, 12])):
    cell = ws_det.cell(1, 1 + j, t); cell.font = F_TIT; cell.fill = FILL_H
    ws_det.column_dimensions[L(1 + j)].width = w
if not det.empty:
    det = det.sort_values(["DEPARTAMENTO", "BLOQUE", "COMPARACIÓN", "DIFERENCIA S/"], ascending=[True, True, True, False])
    for i, fila in enumerate(det[cols_det].itertuples(index=False), start=2):
        for j, v in enumerate(fila):
            cell = ws_det.cell(i, 1 + j, None if (isinstance(v, float) and np.isnan(v)) else v); cell.font = F_NOR
            if 6 <= j <= 8:
                cell.number_format = "#,##0"
            elif j == 9:
                cell.number_format = "0.0"
    ws_det.auto_filter.ref = f"A1:{L(len(cols_det))}{len(det) + 1}"
ws_det.freeze_panes = "A2"

# Hoja RESUMEN
ws_res["A1"] = "VARIACIONES ALARMANTES DE LA INVERSIÓN PÚBLICA · LORETO, SAN MARTÍN Y UCAYALI"; ws_res["A1"].font = F_TIT
lineas = [
    f"Generado: {date.today().isoformat()} · último mes con datos: {etq_mes(u)}",
    "Dato: devengado de la genérica 2.6 Adquisición de activos no financieros (MEF, Consulta Amigable), en soles corrientes.",
    f"Criterio: los {VARIACIONES_TOP} mayores aumentos y las {VARIACIONES_TOP} mayores caídas en MONTO (variación en soles, no porcentual).",
    "Bloques por departamento: total por nivel de gobierno (y de los tres juntos), todos los niveles juntos (nivel y función), "
    "cada nivel por función y los gobiernos locales por municipalidad.",
    "Comparaciones: " + " · ".join(f"{c[0]} ({etq_periodo(c[2])} vs {etq_periodo(c[1])})" for c in comparaciones),
    "Diferencia S/ = periodo actual - periodo anterior. Var. % es solo referencial; 'n.a.' = sin monto en el periodo anterior. "
    "La hoja DETALLE trae todas las variaciones para filtrar.",
    ("16: Comunicaciones se suma en 15: Transporte, como en los cuadros." if AGRUPAR_COMUNICACIONES_EN_TRANSPORTE else ""),
]
for i, t in enumerate([x for x in lineas if x], start=3):
    ws_res.cell(i, 1, t).font = F_NOR
r = i + 2
ws_res.cell(r, 1, "Avisos").font = F_TIT
for j, t in enumerate(notas_var or ["Sin avisos: todos los niveles tienen datos en los periodos comparados."], start=1):
    ws_res.cell(r + j, 1, t).font = F_BAJA if notas_var else F_NOR
ws_res.column_dimensions["A"].width = 130

nombre_var = f"{VARIACIONES_NOMBRE} {u:%Y-%m}.xlsx"
ruta_var = os.path.join("/content", nombre_var)
wb.save(ruta_var)

def subir_xlsx_a_drive(ruta_local, folder_id, max_reintentos=5):
    """Crea o actualiza (por nombre) el archivo en la carpeta de Drive. Requiere permiso de Editor."""
    import time
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaFileUpload
    auth.authenticate_user()
    servicio = build("drive", "v3", cache_discovery=False)
    carpeta = servicio.files().get(fileId=folder_id, fields="name,mimeType,capabilities(canAddChildren)",
                                   supportsAllDrives=True).execute()
    if carpeta.get("mimeType") != "application/vnd.google-apps.folder":
        raise ValueError(f"El ID {folder_id} no corresponde a una carpeta de Drive.")
    if not carpeta.get("capabilities", {}).get("canAddChildren", False):
        raise PermissionError(f"Sin permiso de Editor en '{carpeta['name']}'.")
    nombre = os.path.basename(ruta_local)
    nombre_q = nombre.replace("\\", "\\\\").replace("'", "\\'")
    mime = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    for intento in range(1, max_reintentos + 1):
        try:
            existentes = servicio.files().list(q=f"name = '{nombre_q}' and '{folder_id}' in parents and trashed = false",
                                               fields="files(id)", supportsAllDrives=True,
                                               includeItemsFromAllDrives=True).execute().get("files", [])
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if existentes:
                servicio.files().update(fileId=existentes[0]["id"], media_body=media, supportsAllDrives=True).execute()
                return f"actualizado en '{carpeta['name']}'"
            servicio.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                    fields="id", supportsAllDrives=True).execute()
            return f"creado en '{carpeta['name']}'"
        except HttpError as e:
            if e.resp.status not in {403, 429, 500, 502, 503, 504} or intento == max_reintentos:
                raise
            time.sleep(2 ** intento)

try:
    print(f"☁️  {nombre_var}: {subir_xlsx_a_drive(ruta_var, VARIACIONES_FOLDER_ID)}")
except Exception as e:
    import shutil
    respaldo = os.path.join(RUTA_SALIDA, nombre_var)
    shutil.copy(ruta_var, respaldo)
    print(f"❌ No se pudo subir a la carpeta {VARIACIONES_FOLDER_ID}: {e}\n   Copia guardada en {respaldo}")

if notas_var:
    print("⚠️ " + "\n⚠️ ".join(notas_var))
if not det.empty:
    print(f"\n{TITULO_TOTALES} · {comparaciones[0][0]} (soles):")
    vista = pd.DataFrame([x for x in detalle if x["BLOQUE"] == TITULO_TOTALES and x["COMPARACIÓN"] == comparaciones[0][0]])
    display(vista[["DEPARTAMENTO", "ÍTEM", "PERIODO ANTERIOR", "ANTERIOR S/", "PERIODO ACTUAL", "ACTUAL S/", "DIFERENCIA S/"]]
            .rename(columns={"ÍTEM": "NIVEL"})
            .sort_values("DEPARTAMENTO", kind="stable").reset_index(drop=True))
    print(f"\nTodos los niveles · {comparaciones[0][0]} · {VARIACIONES_TOP} mayores aumentos y caídas (soles):")
    vista = det[(det["BLOQUE"] == BLOQUES_VAR[0][0]) & (det["COMPARACIÓN"] == comparaciones[0][0]) & (det["RANKING"] != "")]
    display(vista[["DEPARTAMENTO", "RANKING", "ÍTEM", "ANTERIOR S/", "ACTUAL S/", "DIFERENCIA S/"]]
            .sort_values(["DEPARTAMENTO", "RANKING"]).reset_index(drop=True))
